# Create and Update Job Properties #
This sample will create a job with job properties, extended properties. Then it will show examples of how to update each. 

In [7]:
import arcgis
import re
import datetime
from arcgis.gis.workflowmanager import WorkflowManager

gis = arcgis.gis.GIS(profile="ps0008227", verify_cert=False)
item = gis.content.search('title:"Python Sample"')[0]
workflowManager = WorkflowManager(item)


## Create Job Template with Extended Properties

In [11]:
# Get Default diagram
diagrams = workflowManager.diagrams
diagram = {}
for gram in diagrams:
    if gram.diagram_name == 'Introduction to Workflow Manager':
        diagram = gram
        break
            
# Create the job template with extended properties 
uniqueness = re.sub("[^0-9a-z]+", "_", str(datetime.datetime.now()))

#unique names so there is no duplicate naming errors
template_name = 'Example Template  ' + uniqueness
table_name = 'example_table_' + uniqueness

success = workflowManager.create_job_template(name=template_name, diagram_id=diagram.diagram_id, diagram_name=diagram.diagram_name,
                                        priority='High', category='Functional Tests', job_duration=5,assigned_to="admin", 
                                        default_due_date='2020-04-02T13:25:50Z', default_start_date='2020-04-02T13:25:50Z',
                                        start_date_type='CreationDate',assigned_type='Unassigned', description='Test Test test',
                                        default_description='Test Test123', state='Active', last_updated_by='Abbie Admin',
                                        last_updated_date='2020-04-02T13:25:50Z',
                                        extended_property_table_definitions=[
                                            {
                                                "tableName": table_name,
                                                "tableAlias": table_name,
                                                "tableOrder": 0,
                                                "relationshipType": "OneToOne",
                                                "extendedPropertyDefinitions": [
                                                    # String Property
                                                    {"propertyOrder": 0,
                                                     "visible": True,
                                                     "propertyName": "prop1",
                                                     "editable": True,
                                                     "dataType": "String",
                                                     "propertyAlias": "prop1",
                                                     "required": True,
                                                     "fieldLength": 50
                                                     },
                                                    # String Property
                                                    {"propertyOrder": 1,
                                                     "visible": True,
                                                     "propertyName": "prop2",
                                                     "editable": True,
                                                     "dataType": "String",
                                                     "propertyAlias": "prop2",
                                                     "required": True,
                                                     "fieldLength": 50
                                                     },
                                                    # Domain Property
                                                    {"propertyOrder": 2,
                                                     "visible": True,
                                                     "propertyName": "prop4",
                                                     "editable": True,
                                                     "domain": {
                                                         "type": "codedValue",
                                                         "codedValues": [
                                                             {
                                                                 "code": "1",
                                                                 "name": "Uno"
                                                             },
                                                             {
                                                                 "code": "2",
                                                                 "name": "Dos"
                                                             }
                                                         ],
                                                         "range": [
                                                             "string"
                                                         ]
                                                     },
                                                     "dataType": "Integer",
                                                     "propertyAlias": "prop4",
                                                     "required": True,
                                                     "fieldLength": 50
                                                     }
                                                ],
                                            }
                                        ]
                                        )
if success:
    print('Successfully created template')

Successfully created template


## Create Job

In [17]:
# Find newly create template
job_templates = workflowManager.job_templates
job_template = {}
for x in job_templates:
    if x.job_template_name == template_name:
        job_template = x
        
print('Template \"' + job_template.job_template_name + '\" Id: ' + job_template.job_template_id)
        
# Create Job setting some of the jobs extended properties
job_ids = workflowManager.jobs.create( template=job_template.job_template_id, count=1, name='Test New Job123',
                                        start='2020-04-02T13:25:50Z', end='2020-04-02T13:25:50Z', priority='High',
                                        description='Perfect description of work', owner="admin", assigned="admin",
                                        complete=42, notes='testing notes', parent='',
                                        extended_properties=[
                                            # String Property
                                            {
                                                "identifier": table_name + ".prop1",
                                                "value": "newly_created123"
                                            },
                                            # String Property
                                            {
                                                "identifier": table_name + ".prop2",
                                                "value": "newly_created456"
                                            },
                                            # Domain Property
                                            {
                                                "identifier": table_name + ".prop4",
                                                "value": "1"
                                            }

                                        ])
print('Created job: ' + job_ids[0])

Template "Example Template  2021_05_11_13_21_17_783922" Id: t98IsJlPRq6fqAEDuq3Hkg
Created: 5facuDmqRW-Tk3Cvcl05bQ


## Update Job Properties


In [22]:
# Get job we previously created. Set True to return the extended properties with the get call.
job = workflowManager.jobs.get(job_ids[0], True)

#update job properties 
job.priority = 'Updated'
job.job_name = 'updated Job 123'
job.description = "Updated description"

# update extended properties
table_name = job.extended_properties[0]["tableName"]
job.extended_properties = [
    {
        "identifier": table_name + ".prop1",
        "value": "updated_123"
    },
    {
        "identifier": table_name + ".prop2",
        "value": "updated_456"
    },
    {
        "identifier": table_name + ".prop4",
        "value": "2"
    },
]

# Actual update call
success = workflowManager.jobs.update(job_ids[0], vars(job))

if success:
    print('Successfully updated template\n')

updated_job = workflowManager.jobs.get(job_ids[0], True)

print('Updated Job\'s Extended Properties')
print(updated_job.extended_properties)

Successfully updated template

Updated Job's Extended Properties
[{'tableName': 'example_table_2021_05_11_13_21_17_783922', 'properties': [{'propertyName': 'prop1', 'value': 'updated_123'}, {'propertyName': 'prop2', 'value': 'updated_456'}, {'propertyName': 'prop4', 'value': 2}]}]
